In [1]:
import random
import json
import numpy as np

# Round down to nearest divisor
def round_down(num, divisor):
    return num - (num%divisor)

# Custom JSON encoder to maintain integer keys
class IntKeyDictEncoder(json.JSONEncoder):
    def encode(self, obj):
        if isinstance(obj, dict):
            return '{' + ', '.join(f'{k}: {self.encode(v)}' for k, v in obj.items()) + '}'
        return json.JSONEncoder.encode(self, obj)
    
def generate_jobs_from_duration_sets(num_jobs, num_machines, duration_set_0, duration_set_1, max_makespan, slack):
    jobs = {}
    job_count_per_machine = num_jobs // num_machines
    inv_slack = 1-slack

    # Create jobs for machine 0
    total_duration_0 = 0
    for i in range(job_count_per_machine):
        duration = duration_set_0[i]
        release_ = round_down(total_duration_0, 10)
        release = random.choices([0, release_], weights=[slack, inv_slack])[0]
        # deadline_ = min(release + duration + random.randint(10, 50), max_makespan)
        deadline_ = release + duration
        deadline = random.choices([max_makespan, deadline_], weights=[slack, inv_slack])[0]
        jobs[i + 1] = {"duration": duration, "release": release, "deadline": deadline, "machine": 0}
        total_duration_0 += duration

    # Create jobs for machine 1
    total_duration_1 = 0
    for i in range(job_count_per_machine):
        duration = duration_set_1[i]
        release_ = round_down(total_duration_1, 10)
        release = random.choices([0, release_], weights=[slack, inv_slack])[0]
        # deadline_ = min(release + duration + random.randint(10, 50), max_makespan)
        deadline_ = release + duration
        deadline = random.choices([max_makespan, deadline_], weights=[slack, inv_slack])[0]
        jobs[i + job_count_per_machine + 1] = {"duration": duration, "release": release, "deadline": deadline, "machine": 1}
        total_duration_1 += duration

    makespan = total_duration_0  # Assuming the makespans are already balanced
    return jobs, [makespan, makespan]

def calculate_slack(data):
    # Determine the minimum release value and maximum deadline value in the dictionary
    min_release = min(item['release'] for item in data.values())
    max_deadline = max(item['deadline'] for item in data.values())

    # Total number of entries in the dictionary
    total_entries = len(data)
    
    # Count the number of entries with release == min_release and deadline == max_deadline
    count_matching_entries = sum(
        1 for item in data.values() if item['release'] == min_release and item['deadline'] == max_deadline
    )
    
    # Calculate the percentage
    percentage = (count_matching_entries / total_entries) * 100
    
    print(f'Problem instance has {percentage}% slackness')

In [2]:
def generate_jobs_from_duration_sets(num_jobs, num_machines, duration_set_0, duration_set_1, max_makespan, slack_jobs, buffer):
    jobs = {}
    job_count_per_machine = num_jobs // num_machines

    # Calculate the number of jobs that should have maximum slack
    slack_jobs_count = int(slack_jobs * num_jobs)
    non_slack_jobs_count = num_jobs - slack_jobs_count
    
    # Create jobs for machine 0
    total_duration_0 = 0
    for i in range(job_count_per_machine):
        duration = duration_set_0[i]
        release_ = round_down(total_duration_0, 10)

        # Determine if this job should contribute to slack or not
        if i < non_slack_jobs_count // 2:
            # Normal job, doesn't contribute to slack
            release = int(release_ * 1)
            deadline_ = release + duration
            deadline = min(int(deadline_ * (1+4)), max_makespan)
        else:
            # Slack job
            release = 0
            deadline = max_makespan

        jobs[i + 1] = {"duration": duration, "release": release, "deadline": deadline, "machine": 0}
        total_duration_0 += duration

    # Create jobs for machine 1
    total_duration_1 = 0
    for i in range(job_count_per_machine):
        duration = duration_set_1[i]
        release_ = round_down(total_duration_1, 10)

        # Determine if this job should contribute to slack or not
        if i < non_slack_jobs_count // 2:
            # Normal job, doesn't contribute to slack
            release = int(release_ * 1)
            deadline_ = release + duration
            deadline = min(int(deadline_ * (1+4)), max_makespan)
        else:
            # Slack job
            release = 0
            deadline = max_makespan

        jobs[i + job_count_per_machine + 1] = {"duration": duration, "release": release, "deadline": deadline, "machine": 1}
        total_duration_1 += duration

    makespan = max(total_duration_0, total_duration_1)
    return jobs, [makespan, makespan]

proven

In [ ]:
def generate_jobs_from_duration_sets(num_jobs, num_machines, duration_set_0, duration_set_1, max_makespan, slack_jobs, tf=0.4, rdd=0.2):
    jobs = {}
    job_count_per_machine = num_jobs // num_machines

    # Calculate the number of jobs that should have maximum slack
    slack_jobs_count = int(slack_jobs * num_jobs)
    non_slack_jobs_count = num_jobs - slack_jobs_count

    # Compute total duration (P) and the due date range
    P = sum(duration_set_0) + sum(duration_set_1)
    min_due = P * (1 - tf - rdd / 2)
    max_due = P * (1 - tf + rdd / 2)
    
    # Create jobs for machine 0
    total_duration_0 = 0
    for i in range(job_count_per_machine):
        duration = duration_set_0[i]
        release_ = round_down(total_duration_0, 10)

        # Determine if this job should contribute to slack or not
        if i < non_slack_jobs_count // 2:
            # Normal job, doesn't contribute to slack
            release = int(release_ * 1)
            # deadline_ = release + duration
            # deadline = min(int(deadline_ * (1+4)), max_makespan)
            deadline = int(np.random.uniform(min_due, max_due))

        else:
            # Slack job
            release = 0
            deadline = int(np.random.uniform(min_due, max_due))

        jobs[i + 1] = {"duration": duration, "release": release, "deadline": deadline, "machine": 0}
        total_duration_0 += duration

    # Create jobs for machine 1
    total_duration_1 = 0
    for i in range(job_count_per_machine):
        duration = duration_set_1[i]
        release_ = round_down(total_duration_1, 10)

        # Determine if this job should contribute to slack or not
        if i < non_slack_jobs_count // 2:
            # Normal job, doesn't contribute to slack
            release = int(release_ * 1)
            # deadline_ = release + duration
            # deadline = min(int(deadline_ * (1+4)), max_makespan)
            deadline = int(np.random.uniform(min_due, max_due))
            
        else:
            # Slack job
            release = 0
            deadline = int(np.random.uniform(min_due, max_due))

        jobs[i + job_count_per_machine + 1] = {"duration": duration, "release": release, "deadline": deadline, "machine": 1}
        total_duration_1 += duration

    makespan = max(total_duration_0, total_duration_1)
    return jobs, [makespan, makespan]

tf rdd

In [116]:
def generate_jobs_from_duration_sets(num_jobs, num_machines, duration_set_0, duration_set_1, tf=0.1, rdd=0.8, release_factor=0.8):
    jobs = {}
    job_count_per_machine = num_jobs // num_machines

    # Compute total duration (P), due date range, and release date
    P = sum(duration_set_0) + sum(duration_set_1)
    Rmax = release_factor * P
    min_due = P * (1 - tf - rdd / 2)
    max_due = P * (1 - tf + rdd / 2)

    # Create jobs for machine 0
    total_duration_0 = 0
    for i in range(job_count_per_machine):
        duration = duration_set_0[i]
        release = np.random.randint(0, int(Rmax))
        # release = 0
        deadline = int(np.random.uniform(min_due, max_due))
        jobs[i + 1] = {"duration": duration, "release": release, "deadline": deadline, "machine": 0}
        total_duration_0 += duration

    # Create jobs for machine 1
    total_duration_1 = 0
    for i in range(job_count_per_machine):
        duration = duration_set_1[i]
        release = np.random.randint(0, int(Rmax))
        # release = 0
        deadline = int(np.random.uniform(min_due, max_due))
        jobs[i + job_count_per_machine + 1] = {"duration": duration, "release": release, "deadline": deadline, "machine": 1}
        total_duration_1 += duration

    makespan = total_duration_0  # Assuming the makespans are already balanced
    return jobs, [makespan, makespan]

tf rdd preserve order

In [ ]:
import numpy as np

def generate_jobs_from_duration_sets(num_jobs, num_machines, duration_set_0, duration_set_1, tf=0.2, rdd=0.4, release_factor=0.2):
    jobs = {}
    job_count_per_machine = num_jobs // num_machines

    # Compute total duration (P), due date range, and release window
    P = sum(duration_set_0) + sum(duration_set_1)
    Rmax = release_factor * P
    min_due = P * (1 - tf - rdd / 2)
    max_due = P * (1 - tf + rdd / 2)

    # Machine 0
    current_time_0 = 0
    for i in range(job_count_per_machine):
        duration = duration_set_0[i]
        # release = current_time_0
        release = current_time_0 + np.random.randint(0, int(Rmax))
        # Expected finish: release + duration
        est_finish = release + duration
        # Deadline loosely based on finish time, shifted using TF/RDD window
        deadline = int(np.random.uniform(est_finish + min_due * 0.01, est_finish + max_due * 0.01))
        jobs[i + 1] = {
            "duration": duration,
            "release": int(release),
            "deadline": max(int(deadline), int(release + duration)),  # ensure feasible
            "machine": 0
        }
        current_time_0 += duration  # maintain order

    # Machine 1
    current_time_1 = 0
    for i in range(job_count_per_machine):
        duration = duration_set_1[i]
        # release = current_time_1
        release = current_time_1 + np.random.randint(0, int(Rmax))
        est_finish = release + duration
        deadline = int(np.random.uniform(est_finish + min_due * 0.01, est_finish + max_due * 0.01))
        jobs[i + job_count_per_machine + 1] = {
            "duration": duration,
            "release": int(release),
            "deadline": max(int(deadline), int(release + duration)),
            "machine": 1
        }
        current_time_1 += duration

    makespan = max(current_time_0, current_time_1)
    return jobs, [makespan, makespan]


tf rdd preserve machine-level division

In [ ]:
def generate_jobs_from_duration_sets(num_jobs, num_machines, duration_set_0, duration_set_1,
                                     tf=0.4, rdd=0.2, release_factor=0.5):
    jobs = {}
    job_count_per_machine = num_jobs // num_machines

    # Total duration (P) across both machines
    P = sum(duration_set_0) + sum(duration_set_1)

    # Shared due date window (based on TF & RDD)
    min_due_global = P * (1 - tf - rdd / 2)
    max_due_global = P * (1 - tf + rdd / 2)

    # Max release time
    Rmax = release_factor * P

    # Machine-specific time windows (to preserve job grouping)
    mid_point = P // 2
    machine_windows = {
        0: (0, mid_point),
        1: (mid_point, P)
    }

    # Create jobs for machine 0 and 1
    for machine_id, durations in enumerate([duration_set_0, duration_set_1]):
        start, end = machine_windows[machine_id]

        for i, duration in enumerate(durations):
            # Random release within machine-specific window
            # release = np.random.randint(start, min(end, start + int(Rmax)))
            release = np.random.randint(start, start + 10)
            # Due date drawn globally, then made feasible
            due_date = int(np.random.uniform(min_due_global, max_due_global))
            deadline = max(due_date, release + duration + 10)  # ensure feasible but tight

            job_id = i + 1 + machine_id * job_count_per_machine
            jobs[job_id] = {
                "duration": duration,
                "release": release,
                "deadline": deadline,
                "machine": machine_id
            }

    makespan = max(sum(duration_set_0), sum(duration_set_1))
    return jobs, [makespan, makespan]

tf rdd preserve machine-level division
- Make release times start early (e.g. within 20% of the machine's local time window)
- Still assign jobs to machine-specific windows for separation

In [421]:
def generate_jobs_from_duration_sets(num_jobs, num_machines, duration_set_0, duration_set_1,
                                     tf=0.4, rdd=0.2, release_factor=0.1):
    jobs = {}
    job_count_per_machine = num_jobs // num_machines

    # Total duration (P) across both machines
    P = sum(duration_set_0) + sum(duration_set_1)

    # Softer due date range
    min_due_global = P * (1 - tf - rdd / 2)
    max_due_global = P * (1 - tf + rdd / 2)

    # Machine-specific time windows (for gentler partitioning)
    mid_point = P // 2
    machine_windows = {
        0: (0, mid_point),
        1: (mid_point, P)
    }

    # Create jobs for machine 0 and 1
    for machine_id, durations in enumerate([duration_set_0, duration_set_1]):
        start, end = machine_windows[machine_id]
        local_P = sum(durations)

        for i, duration in enumerate(durations):
            # Easier release: early in the local window
            release = np.random.randint(start, start + max(int(release_factor * local_P), 1))
            # release = start

            # Due date drawn globally, then made feasible
            due_date = int(np.random.uniform(min_due_global, max_due_global))
            deadline = max(due_date, release + duration + 10)  # ensure feasible but tight

            job_id = i + 1 + machine_id * job_count_per_machine
            jobs[job_id] = {
                "duration": duration,
                "release": release,
                "deadline": deadline,
                "machine": machine_id
            }

    makespan = max(sum(duration_set_0), sum(duration_set_1))
    return jobs, [makespan, makespan]


tf rdd preserve machine-level division II
- Make release times start early (e.g. within 20% of the machine's local time window)
- Still assign jobs to machine-specific windows for separation

In [422]:
# def generate_jobs_from_duration_sets(num_jobs, num_machines, duration_set_0, duration_set_1,
#                                      tf=0.4, rdd=0.2, release_factor=0.1):
#     jobs = {}
#     job_count_per_machine = num_jobs // num_machines

#     # Total duration (P) across both machines
#     P = sum(duration_set_0) + sum(duration_set_1)
#     mid_point = P // 2

#     # Softer due date range
#     min_due_0 = mid_point * (1 - tf - rdd / 2)
#     max_due_0 = mid_point * (1 - tf + rdd / 2)

#     min_due_1 = P * (1 - tf - rdd / 2)
#     max_due_1 = P * (1 - tf + rdd / 2)

#     machine_due_windows = {
#         0: (min_due_0, max_due_0),
#         1: (min_due_1, max_due_1)
#     }

#     # Machine-specific time windows (for gentler partitioning)
#     machine_release_windows = {
#         0: (0, mid_point),
#         1: (mid_point, P)
#     }
    

#     # Create jobs for machine 0 and 1
#     for machine_id, durations in enumerate([duration_set_0, duration_set_1]):
#         start_release, end_release = machine_release_windows[machine_id]
#         start_due, end_due = machine_due_windows[machine_id]
#         local_P = sum(durations)

#         for i, duration in enumerate(durations):
#             # Easier release: early in the local window
#             release = np.random.randint(start_release, start_release + int(release_factor * local_P))
#             # release = start

#             # Due date drawn globally, then made feasible
#             due_date = int(np.random.uniform(start_due, end_due))
#             deadline = max(due_date, release + duration + 10)  # ensure feasible but tight

#             job_id = i + 1 + machine_id * job_count_per_machine
#             jobs[job_id] = {
#                 "duration": duration,
#                 "release": release,
#                 "deadline": deadline,
#                 "machine": machine_id
#             }

#     makespan = max(sum(duration_set_0), sum(duration_set_1))
#     return jobs, [makespan, makespan]


In [423]:
# under 5% overlap of the total possible combinations

# ([19, 1], [18, 2]) #4 jobs
# ([20, 18, 1, 1], [19, 15, 3, 3]) #8 jobs
# ([20, 20, 20, 16, 1, 1, 1, 1], [19, 19, 19, 11, 4, 3, 3, 2]) #16 jobs
# ([20, 20, 20, 20, 6, 1, 1, 1, 1], [19, 19, 19, 17, 7, 3, 2, 2, 2]) #18 jobs
# ([20, 20, 20, 20, 15, 1, 1, 1, 1, 1], [19, 19, 19, 19, 10, 4, 3, 3, 2, 2]) #20 jobs
# ([20, 20, 20, 20, 20, 14, 1, 1, 1, 1, 1, 1], [19, 19, 19, 19, 19, 9, 4, 4, 2, 2, 2, 2]) #22 jobs

In [424]:
# num_machines = 2
# duration_set_0 = [134,
#   140,
#   68,
#   165,
#   49,
#   119,
#   189,
#   175,
#   48,
#   216,
#   191,
#   56,
#   203,
#   193,
#   240,
#   70,
#   49,
#   148,
#   204,
#   94,
#   113,
#   184,
#   147,
#   139,
#   114,
#   145,
#   213,
#   65,
#   199,
#   92,
#   139,
#   176,
#   76,
#   237,
#   82,
#   197,
#   220,
#   197,
#   190,
#   142,
#   237,
#   171,
#   66,
#   173,
#   101,
#   236,
#   220,
#   236,
#   73,
#   4,
#   33,
#   141,
#   59,
#   18,
#   139,
#   200,
#   90,
#   250,
#   219,
#   25,
#   131,
#   16,
#   89,
#   68,
#   169,
#   168,
#   176,
#   104,
#   209,
#   244,
#   68,
#   188,
#   216,
#   249,
#   128,
#   82,
#   46,
#   26,
#   118,
#   160,
#   28,
#   163,
#   75,
#   108,
#   236,
#   21,
#   52,
#   227,
#   87,
#   126,
#   144,
#   23,
#   17,
#   215,
#   99,
#   127,
#   167,
#   152,
#   51,
#   42]
# duration_set_1 = [159,
#   188,
#   149,
#   183,
#   12,
#   241,
#   114,
#   54,
#   12,
#   207,
#   175,
#   132,
#   246,
#   93,
#   110,
#   242,
#   232,
#   49,
#   150,
#   168,
#   54,
#   220,
#   159,
#   68,
#   245,
#   116,
#   69,
#   163,
#   143,
#   61,
#   83,
#   70,
#   89,
#   117,
#   115,
#   66,
#   151,
#   158,
#   170,
#   13,
#   120,
#   91,
#   114,
#   82,
#   123,
#   83,
#   144,
#   166,
#   215,
#   47,
#   131,
#   192,
#   177,
#   98,
#   143,
#   232,
#   218,
#   122,
#   38,
#   115,
#   242,
#   40,
#   102,
#   90,
#   4,
#   47,
#   140,
#   19,
#   228,
#   37,
#   107,
#   62,
#   251,
#   100,
#   225,
#   125,
#   148,
#   206,
#   107,
#   208,
#   234,
#   172,
#   167,
#   56,
#   195,
#   50,
#   187,
#   240,
#   5,
#   225,
#   47,
#   211,
#   156,
#   200,
#   228,
#   87,
#   184,
#   83,
#   205,
#   7]

# num_jobs = len(duration_set_0) + len(duration_set_1)
# slack = 0.6
# buffer = 1
# slack_str = int(slack*100)
# json_path = f'../data_gecco/ssjsp/ssjsp_{num_jobs}_s{slack_str}_tfrdd.json'
# max_makespan = sum(duration_set_0)

# # jobs, makespans = generate_jobs_from_duration_sets(num_jobs, num_machines, duration_set_0, duration_set_1, max_makespan, slack, buffer)
# jobs, makespans = generate_jobs_from_duration_sets(num_jobs, num_machines, duration_set_0, duration_set_1, tf=0.2, rdd=0.4, release_factor=0.2)

In [425]:
# num_machines = 2
# duration_set_0 = [141,
#   205,
#   43,
#   22,
#   150,
#   221,
#   173,
#   121,
#   179,
#   118,
#   14,
#   181,
#   3,
#   114,
#   109,
#   42,
#   234,
#   137,
#   180,
#   18,
#   67,
#   208,
#   16,
#   203,
#   57,
#   135,
#   87,
#   145,
#   19,
#   73,
#   169,
#   28,
#   73,
#   213,
#   186,
#   61,
#   14,
#   107,
#   166,
#   93,
#   238,
#   140,
#   31,
#   242,
#   120,
#   76,
#   233,
#   195,
#   89,
#   197,
#   103,
#   11,
#   218,
#   248,
#   75,
#   227,
#   106,
#   192,
#   97,
#   34,
#   78,
#   231,
#   218,
#   25,
#   180,
#   135,
#   214,
#   81,
#   188,
#   163,
#   251,
#   249,
#   16,
#   105,
#   247,
#   159,
#   189,
#   102,
#   248,
#   46,
#   87,
#   105,
#   205,
#   166,
#   157,
#   130,
#   236,
#   177,
#   99,
#   68,
#   2,
#   236,
#   112,
#   62,
#   52,
#   90,
#   181,
#   80,
#   1,
#   171,
#   52,
#   164,
#   220,
#   85,
#   9,
#   123,
#   117,
#   224,
#   219,
#   193,
#   81,
#   240,
#   11,
#   183,
#   173,
#   61,
#   100,
#   174,
#   48,
#   39,
#   110,
#   30,
#   39,
#   22,
#   74,
#   56,
#   16,
#   179,
#   118,
#   128,
#   148,
#   114,
#   184,
#   136,
#   155,
#   71,
#   115,
#   183,
#   176,
#   2,
#   233,
#   122,
#   96,
#   45,
#   18,
#   254,
#   138,
#   157,
#   91,
#   235,
#   167,
#   254,
#   126,
#   218,
#   63,
#   45,
#   55,
#   11,
#   180,
#   167,
#   27,
#   161,
#   5,
#   69,
#   89,
#   157,
#   183,
#   175,
#   205,
#   90,
#   14,
#   199,
#   14,
#   38,
#   78,
#   97,
#   131,
#   118,
#   7,
#   96,
#   18,
#   253,
#   224,
#   204,
#   6,
#   144,
#   110,
#   109,
#   31,
#   123,
#   154,
#   95,
#   61,
#   188,
#   254,
#   118,
#   246,
#   248,
#   201,
#   6,
#   16,
#   55,
#   44,
#   164,
#   4,
#   141,
#   234,
#   5,
#   139,
#   239,
#   12,
#   168,
#   192,
#   41,
#   50,
#   62,
#   230,
#   148,
#   223,
#   32,
#   218,
#   21,
#   200,
#   142,
#   40,
#   217,
#   33,
#   5,
#   24,
#   173,
#   160,
#   202,
#   179,
#   178,
#   128,
#   15,
#   128,
#   138,
#   231,
#   174,
#   248,
#   76,
#   15,
#   164,
#   30,
#   253,
#   203,
#   22,
#   191,
#   75]
# duration_set_1 = [93,
#   130,
#   94,
#   250,
#   233,
#   21,
#   153,
#   242,
#   96,
#   7,
#   79,
#   40,
#   230,
#   178,
#   121,
#   49,
#   146,
#   50,
#   148,
#   172,
#   154,
#   173,
#   231,
#   5,
#   61,
#   47,
#   120,
#   253,
#   158,
#   98,
#   230,
#   179,
#   10,
#   118,
#   137,
#   225,
#   50,
#   182,
#   199,
#   20,
#   158,
#   182,
#   172,
#   28,
#   5,
#   89,
#   47,
#   149,
#   214,
#   101,
#   232,
#   89,
#   167,
#   156,
#   187,
#   77,
#   124,
#   180,
#   203,
#   144,
#   31,
#   172,
#   229,
#   185,
#   63,
#   228,
#   76,
#   13,
#   70,
#   105,
#   38,
#   147,
#   196,
#   236,
#   166,
#   34,
#   17,
#   126,
#   39,
#   56,
#   47,
#   156,
#   198,
#   176,
#   164,
#   21,
#   59,
#   37,
#   117,
#   89,
#   42,
#   198,
#   247,
#   25,
#   152,
#   109,
#   173,
#   160,
#   105,
#   6,
#   180,
#   218,
#   161,
#   24,
#   144,
#   10,
#   60,
#   177,
#   60,
#   151,
#   120,
#   32,
#   187,
#   32,
#   90,
#   56,
#   5,
#   28,
#   252,
#   187,
#   114,
#   218,
#   143,
#   114,
#   231,
#   83,
#   15,
#   169,
#   78,
#   250,
#   53,
#   8,
#   98,
#   113,
#   135,
#   2,
#   23,
#   8,
#   161,
#   216,
#   195,
#   80,
#   254,
#   50,
#   193,
#   135,
#   181,
#   43,
#   82,
#   13,
#   143,
#   173,
#   184,
#   156,
#   149,
#   13,
#   29,
#   169,
#   73,
#   72,
#   204,
#   92,
#   27,
#   68,
#   52,
#   97,
#   213,
#   152,
#   60,
#   177,
#   217,
#   191,
#   105,
#   188,
#   105,
#   26,
#   9,
#   58,
#   239,
#   43,
#   129,
#   221,
#   187,
#   101,
#   192,
#   238,
#   236,
#   158,
#   55,
#   203,
#   203,
#   143,
#   70,
#   197,
#   56,
#   10,
#   234,
#   5,
#   22,
#   109,
#   135,
#   78,
#   176,
#   68,
#   14,
#   187,
#   28,
#   35,
#   117,
#   228,
#   106,
#   74,
#   239,
#   111,
#   92,
#   244,
#   51,
#   186,
#   166,
#   53,
#   127,
#   44,
#   169,
#   175,
#   69,
#   203,
#   73,
#   107,
#   241,
#   246,
#   252,
#   115,
#   76,
#   65,
#   238,
#   138,
#   167,
#   200,
#   242,
#   229,
#   18,
#   17,
#   200,
#   75,
#   111,
#   92,
#   173,
#   1,
#   234,
#   250]

# num_jobs = len(duration_set_0) + len(duration_set_1)
# slack = 0.6
# buffer = 1
# slack_str = int(slack*100)
# json_path = f'../data_gecco/ssjsp/ssjsp_{num_jobs}_s{slack_str}_tfrdd.json'
# max_makespan = sum(duration_set_0)

# # jobs, makespans = generate_jobs_from_duration_sets(num_jobs, num_machines, duration_set_0, duration_set_1, max_makespan, slack, buffer)
# jobs, makespans = generate_jobs_from_duration_sets(num_jobs, num_machines, duration_set_0, duration_set_1, tf=0.2, rdd=0.4, release_factor=0.01)

In [435]:
num_machines = 2
duration_set_0 = [50,
  148,
  38,
  21,
  111,
  153,
  6,
  234,
  6,
  242,
  111,
  203,
  167,
  23,
  39,
  226,
  60,
  214,
  118,
  200,
  220,
  98,
  89,
  223,
  183,
  125,
  219,
  233,
  191,
  14,
  113,
  125,
  202,
  81,
  149,
  172,
  205,
  253,
  234,
  91,
  133,
  104,
  154,
  91,
  13,
  5,
  195,
  127,
  118,
  221,
  159,
  77,
  99,
  101,
  155,
  235,
  220,
  187,
  158,
  82,
  31,
  68,
  53,
  198,
  216,
  234,
  220,
  6,
  117,
  231,
  223,
  153,
  239,
  169,
  219,
  183,
  36,
  210,
  223,
  83,
  191,
  131,
  244,
  186,
  149,
  182,
  254,
  54,
  246,
  94,
  84,
  124,
  160,
  136,
  77,
  57,
  69,
  157,
  192,
  240,
  139,
  154,
  55,
  86,
  181,
  95,
  40,
  87,
  208,
  215,
  186,
  24,
  117,
  178,
  41,
  10,
  161,
  81,
  19,
  139,
  105,
  214,
  195,
  168,
  250,
  187,
  203,
  215,
  69,
  111,
  127,
  191,
  211,
  153,
  151,
  163,
  57,
  65,
  141,
  135,
  175,
  150,
  142,
  17,
  246,
  32,
  229,
  139,
  164,
  98,
  139,
  87,
  86,
  175,
  127,
  201,
  52,
  40,
  139,
  59,
  121,
  235,
  140,
  63,
  129,
  64,
  241,
  90,
  111,
  189,
  252,
  220,
  69,
  166,
  25,
  9,
  247,
  87,
  128,
  67,
  198,
  28,
  192,
  14,
  172,
  155,
  72,
  194,
  72,
  97,
  183,
  201,
  98,
  138,
  142,
  117,
  252,
  86,
  101,
  14,
  133,
  86,
  172,
  254,
  93,
  159,
  216,
  133,
  120,
  142,
  223,
  79,
  73,
  40,
  128,
  173,
  147,
  30,
  165,
  68,
  157,
  71,
  23,
  38,
  158,
  206,
  170,
  176,
  180,
  165,
  113,
  218,
  51,
  150,
  65,
  228,
  33,
  187,
  195,
  167,
  98,
  165,
  209,
  73,
  150,
  243,
  123,
  204,
  4,
  18,
  223,
  217,
  246,
  215,
  135,
  254,
  68,
  53,
  143,
  147,
  143,
  40,
  168,
  100,
  104,
  197,
  94,
  162,
  59,
  185,
  79,
  150,
  99,
  93,
  209,
  140,
  60,
  40,
  116,
  104,
  244,
  210,
  194,
  235,
  75,
  94,
  51,
  130,
  129,
  254,
  169,
  59,
  182,
  94,
  42,
  118,
  232,
  234,
  174,
  34,
  103,
  79,
  6,
  85,
  90,
  252,
  29,
  170,
  68,
  25,
  211,
  234,
  91,
  32,
  199,
  188,
  93,
  106,
  86,
  7,
  94,
  17,
  163,
  17,
  182,
  141,
  121,
  40,
  240,
  195,
  213,
  59,
  128,
  46,
  201,
  160,
  234,
  44,
  147,
  167,
  100,
  122,
  134,
  145,
  231,
  252,
  27,
  163,
  76,
  249,
  148,
  98,
  139,
  144,
  104,
  159,
  177,
  137,
  193,
  66,
  89,
  53,
  156,
  200,
  145,
  242,
  186,
  28,
  72,
  212,
  34,
  105,
  130,
  16,
  76,
  237,
  192,
  154,
  223,
  199,
  191,
  47,
  210,
  253,
  53,
  123,
  117,
  55,
  30,
  35,
  135,
  162,
  82,
  128,
  155,
  227,
  74,
  196,
  7,
  248,
  142,
  229,
  227,
  32,
  8,
  45,
  140,
  208,
  139,
  78,
  172,
  185,
  187,
  157,
  20,
  91,
  197,
  245,
  174,
  36,
  218,
  104,
  198,
  205,
  59,
  87,
  19,
  30,
  141,
  117,
  185,
  2,
  98,
  193,
  38,
  122,
  64,
  137,
  60,
  151,
  226,
  66,
  28,
  99,
  128,
  114,
  228,
  64,
  128,
  145,
  152,
  148,
  201,
  229,
  195,
  73,
  152,
  28,
  43,
  228,
  82,
  239,
  131,
  169,
  121,
  101,
  201,
  146,
  44,
  184,
  188,
  88,
  219,
  210,
  194,
  238]
duration_set_1 = [11,
  201,
  181,
  178,
  161,
  101,
  90,
  142,
  90,
  253,
  101,
  214,
  149,
  210,
  112,
  102,
  211,
  64,
  63,
  37,
  10,
  109,
  135,
  176,
  198,
  84,
  215,
  15,
  196,
  214,
  2,
  5,
  71,
  103,
  240,
  78,
  235,
  163,
  61,
  134,
  75,
  14,
  103,
  89,
  159,
  192,
  137,
  241,
  1,
  155,
  159,
  218,
  25,
  193,
  52,
  104,
  94,
  36,
  122,
  203,
  150,
  17,
  140,
  55,
  173,
  73,
  30,
  48,
  141,
  66,
  18,
  3,
  15,
  13,
  151,
  214,
  121,
  207,
  151,
  75,
  26,
  38,
  36,
  244,
  27,
  228,
  152,
  54,
  240,
  105,
  219,
  201,
  70,
  129,
  214,
  53,
  55,
  56,
  100,
  169,
  132,
  20,
  64,
  85,
  86,
  181,
  87,
  201,
  24,
  128,
  101,
  22,
  65,
  177,
  11,
  244,
  179,
  92,
  81,
  164,
  188,
  216,
  180,
  86,
  160,
  48,
  44,
  2,
  104,
  127,
  189,
  172,
  118,
  176,
  9,
  18,
  5,
  184,
  22,
  240,
  35,
  89,
  33,
  108,
  98,
  243,
  58,
  112,
  29,
  127,
  238,
  91,
  207,
  231,
  198,
  167,
  87,
  61,
  147,
  30,
  79,
  181,
  74,
  144,
  237,
  174,
  91,
  73,
  204,
  117,
  93,
  72,
  225,
  197,
  142,
  86,
  174,
  50,
  148,
  188,
  66,
  222,
  239,
  149,
  74,
  33,
  75,
  31,
  32,
  119,
  63,
  236,
  19,
  172,
  15,
  58,
  17,
  251,
  13,
  185,
  203,
  244,
  85,
  53,
  222,
  78,
  177,
  109,
  190,
  165,
  87,
  32,
  4,
  249,
  140,
  17,
  174,
  125,
  126,
  147,
  32,
  73,
  14,
  86,
  164,
  206,
  187,
  220,
  89,
  81,
  83,
  140,
  38,
  193,
  220,
  188,
  183,
  135,
  31,
  170,
  32,
  65,
  50,
  27,
  109,
  91,
  63,
  128,
  40,
  250,
  95,
  24,
  176,
  236,
  23,
  169,
  246,
  45,
  120,
  104,
  171,
  151,
  184,
  180,
  98,
  103,
  128,
  140,
  95,
  225,
  104,
  93,
  133,
  225,
  163,
  194,
  205,
  54,
  126,
  141,
  16,
  179,
  152,
  146,
  205,
  21,
  228,
  105,
  235,
  238,
  29,
  46,
  45,
  79,
  171,
  120,
  112,
  198,
  76,
  9,
  160,
  129,
  188,
  99,
  204,
  64,
  145,
  79,
  23,
  172,
  212,
  33,
  175,
  157,
  212,
  61,
  164,
  16,
  124,
  180,
  41,
  151,
  107,
  121,
  103,
  90,
  11,
  225,
  157,
  129,
  209,
  24,
  169,
  48,
  22,
  138,
  52,
  186,
  203,
  235,
  187,
  7,
  68,
  53,
  90,
  60,
  223,
  138,
  195,
  109,
  252,
  186,
  82,
  16,
  215,
  97,
  115,
  9,
  243,
  92,
  251,
  170,
  33,
  62,
  254,
  50,
  243,
  56,
  221,
  139,
  10,
  4,
  60,
  250,
  246,
  19,
  92,
  25,
  68,
  169,
  98,
  139,
  212,
  74,
  105,
  190,
  60,
  105,
  207,
  225,
  90,
  179,
  83,
  171,
  194,
  81,
  9,
  130,
  185,
  103,
  31,
  32,
  140,
  57,
  25,
  222,
  12,
  130,
  229,
  27,
  143,
  70,
  121,
  167,
  69,
  63,
  148,
  102,
  182,
  52,
  246,
  217,
  177,
  166,
  51,
  245,
  144,
  96,
  188,
  43,
  33,
  250,
  158,
  92,
  144,
  67,
  210,
  47,
  195,
  236,
  199,
  121,
  30,
  91,
  28,
  116,
  175,
  179,
  160,
  60,
  218,
  94,
  3,
  117,
  217,
  209,
  160,
  17,
  253,
  51,
  74,
  138,
  154,
  207,
  149,
  219,
  216,
  166,
  63,
  59,
  72,
  107,
  19,
  240,
  3,
  119,
  120,
  129,
  231,
  158,
  213,
  21,
  198,
  36,
  36,
  84,
  251,
  174,
  116,
  154,
  94,
  9,
  130,
  61,
  203,
  124,
  230,
  70,
  168,
  50,
  141,
  109,
  200,
  251,
  158,
  49,
  145,
  93,
  156,
  168,
  142,
  73,
  177,
  103,
  209,
  98,
  7,
  47,
  103,
  81,
  5,
  83,
  201,
  222]

num_jobs = len(duration_set_0) + len(duration_set_1)
slack = 0.6
buffer = 1
slack_str = int(slack*100)
json_path = f'../data_gecco/ssjsp/ssjsp_{num_jobs}_s{slack_str}_tfrdd.json'
max_makespan = sum(duration_set_0)

# jobs, makespans = generate_jobs_from_duration_sets(num_jobs, num_machines, duration_set_0, duration_set_1, max_makespan, slack, buffer)
jobs, makespans = generate_jobs_from_duration_sets(num_jobs, num_machines, duration_set_0, duration_set_1, tf=0.2, rdd=0.4, release_factor=0)

In [436]:
P = sum(duration_set_0) + sum(duration_set_1)
P

128216

In [437]:
len(duration_set_0), len(duration_set_1)

(476, 524)

In [428]:
tf = 0.4
rdd = 0.2
min_due = P * (1 - tf - rdd / 2)
max_due = P * (1 - tf + rdd / 2)
print(min_due, max_due)

64108.0 89751.2


In [429]:
num_jobs, len(duration_set_0), len(duration_set_1)

(1000, 476, 524)

In [430]:
jobs, makespans

({1: {'duration': 50, 'release': 0, 'deadline': 113060, 'machine': 0},
  2: {'duration': 148, 'release': 0, 'deadline': 81512, 'machine': 0},
  3: {'duration': 38, 'release': 0, 'deadline': 109761, 'machine': 0},
  4: {'duration': 21, 'release': 0, 'deadline': 82855, 'machine': 0},
  5: {'duration': 111, 'release': 0, 'deadline': 124326, 'machine': 0},
  6: {'duration': 153, 'release': 0, 'deadline': 78410, 'machine': 0},
  7: {'duration': 6, 'release': 0, 'deadline': 121522, 'machine': 0},
  8: {'duration': 234, 'release': 0, 'deadline': 115626, 'machine': 0},
  9: {'duration': 6, 'release': 0, 'deadline': 122787, 'machine': 0},
  10: {'duration': 242, 'release': 0, 'deadline': 115816, 'machine': 0},
  11: {'duration': 111, 'release': 0, 'deadline': 77785, 'machine': 0},
  12: {'duration': 203, 'release': 0, 'deadline': 121823, 'machine': 0},
  13: {'duration': 167, 'release': 0, 'deadline': 85103, 'machine': 0},
  14: {'duration': 23, 'release': 0, 'deadline': 119944, 'machine': 0},


In [431]:
calculate_slack(jobs)

Problem instance has 0.1% slackness


In [432]:
# Save the generated jobs to a JSON file
with open(json_path, 'w') as f:
    json.dump(jobs, f, indent=4, cls=IntKeyDictEncoder)

In [433]:
json_path

'../data_gecco/ssjsp/ssjsp_1000_s60_tfrdd.json'

In [379]:
filename = 'ssjsp_500_s60_tfrdd'
json_path = f'../data_gecco/ssjsp/{filename}.json'

In [380]:
with open(json_path) as f:
    d = json.load(f)
    print(d)

{'1': {'duration': 141, 'release': 10, 'deadline': 43483, 'machine': 0}, '2': {'duration': 205, 'release': 294, 'deadline': 56376, 'machine': 0}, '3': {'duration': 43, 'release': 110, 'deadline': 52456, 'machine': 0}, '4': {'duration': 22, 'release': 126, 'deadline': 51699, 'machine': 0}, '5': {'duration': 150, 'release': 129, 'deadline': 49271, 'machine': 0}, '6': {'duration': 221, 'release': 45, 'deadline': 42225, 'machine': 0}, '7': {'duration': 173, 'release': 146, 'deadline': 42536, 'machine': 0}, '8': {'duration': 121, 'release': 1, 'deadline': 46600, 'machine': 0}, '9': {'duration': 179, 'release': 232, 'deadline': 48149, 'machine': 0}, '10': {'duration': 118, 'release': 0, 'deadline': 46497, 'machine': 0}, '11': {'duration': 14, 'release': 256, 'deadline': 54695, 'machine': 0}, '12': {'duration': 181, 'release': 99, 'deadline': 42731, 'machine': 0}, '13': {'duration': 3, 'release': 72, 'deadline': 59696, 'machine': 0}, '14': {'duration': 114, 'release': 189, 'deadline': 37799, 

In [381]:
d

{'1': {'duration': 141, 'release': 10, 'deadline': 43483, 'machine': 0},
 '2': {'duration': 205, 'release': 294, 'deadline': 56376, 'machine': 0},
 '3': {'duration': 43, 'release': 110, 'deadline': 52456, 'machine': 0},
 '4': {'duration': 22, 'release': 126, 'deadline': 51699, 'machine': 0},
 '5': {'duration': 150, 'release': 129, 'deadline': 49271, 'machine': 0},
 '6': {'duration': 221, 'release': 45, 'deadline': 42225, 'machine': 0},
 '7': {'duration': 173, 'release': 146, 'deadline': 42536, 'machine': 0},
 '8': {'duration': 121, 'release': 1, 'deadline': 46600, 'machine': 0},
 '9': {'duration': 179, 'release': 232, 'deadline': 48149, 'machine': 0},
 '10': {'duration': 118, 'release': 0, 'deadline': 46497, 'machine': 0},
 '11': {'duration': 14, 'release': 256, 'deadline': 54695, 'machine': 0},
 '12': {'duration': 181, 'release': 99, 'deadline': 42731, 'machine': 0},
 '13': {'duration': 3, 'release': 72, 'deadline': 59696, 'machine': 0},
 '14': {'duration': 114, 'release': 189, 'deadl

In [382]:
machine_0_durations = [job['duration'] for job in d.values() if job['machine'] == 0]
machine_1_durations = [job['duration'] for job in d.values() if job['machine'] == 1]

In [383]:
machine_0_durations, machine_1_durations

([141,
  205,
  43,
  22,
  150,
  221,
  173,
  121,
  179,
  118,
  14,
  181,
  3,
  114,
  109,
  42,
  234,
  137,
  180,
  18,
  67,
  208,
  16,
  203,
  57,
  135,
  87,
  145,
  19,
  73,
  169,
  28,
  73,
  213,
  186,
  61,
  14,
  107,
  166,
  93,
  238,
  140,
  31,
  242,
  120,
  76,
  233,
  195,
  89,
  197,
  103,
  11,
  218,
  248,
  75,
  227,
  106,
  192,
  97,
  34,
  78,
  231,
  218,
  25,
  180,
  135,
  214,
  81,
  188,
  163,
  251,
  249,
  16,
  105,
  247,
  159,
  189,
  102,
  248,
  46,
  87,
  105,
  205,
  166,
  157,
  130,
  236,
  177,
  99,
  68,
  2,
  236,
  112,
  62,
  52,
  90,
  181,
  80,
  1,
  171,
  52,
  164,
  220,
  85,
  9,
  123,
  117,
  224,
  219,
  193,
  81,
  240,
  11,
  183,
  173,
  61,
  100,
  174,
  48,
  39,
  110,
  30,
  39,
  22,
  74,
  56,
  16,
  179,
  118,
  128,
  148,
  114,
  184,
  136,
  155,
  71,
  115,
  183,
  176,
  2,
  233,
  122,
  96,
  45,
  18,
  254,
  138,
  157,
  91,
  235,
  167,
  254,

In [38]:
import numpy as np

def generate_jobs_from_duration_sets(
    num_jobs, num_machines, duration_set_0, duration_set_1, max_makespan, slack, resource_tightness
):
    jobs = {}
    job_count_per_machine = num_jobs // num_machines

    # Calculate the number of jobs that should have maximum slack
    slack_jobs_count = int(slack * num_jobs)
    non_slack_jobs_count = num_jobs - slack_jobs_count

    # Create jobs for machine 0
    total_duration_0 = 0
    for i in range(job_count_per_machine):
        duration = duration_set_0[i]
        release_ = round_down(total_duration_0, 10)

        if i < non_slack_jobs_count // 2:
            release = release_
            deadline_ = release + duration
            deadline = min(int(deadline_ * 1.4), max_makespan)
        else:
            release = 0
            deadline = max_makespan

        jobs[i + 1] = {"duration": duration, "release": release, "deadline": deadline, "machine": 0}
        total_duration_0 += duration

    # Create jobs for machine 1
    total_duration_1 = 0
    for i in range(job_count_per_machine):
        duration = duration_set_1[i]
        release_ = round_down(total_duration_1, 10)

        if i < non_slack_jobs_count // 2:
            release = release_
            deadline_ = release + duration
            deadline = min(int(deadline_ * 1.4), max_makespan)
        else:
            release = 0
            deadline = max_makespan

        jobs[i + job_count_per_machine + 1] = {"duration": duration, "release": release, "deadline": deadline, "machine": 1}
        total_duration_1 += duration

    makespan = max(total_duration_0, total_duration_1)

    # Introduce resource constraints
    for job_id, job in jobs.items():
        job["resources"] = np.random.randint(1, resource_tightness + 1)  # Adjust based on tightness

    return jobs, [makespan, makespan]

def compute_sigma_distance(jobs, max_makespan):
    """
    Compute the sigma distance indicator for a given job set.
    """
    # Placeholder for sampling or enumerating solutions
    sampled_makespans = [np.random.randint(1, max_makespan) for _ in range(100)]

    avg_makespan = np.mean(sampled_makespans)
    std_dev = np.std(sampled_makespans)
    reference_makespan = min(sampled_makespans)  # Assuming a heuristic for near-optimal

    sigma_distance = (reference_makespan - avg_makespan) / std_dev if std_dev > 0 else float("inf")
    return sigma_distance

def generate_dataset(num_instances, num_jobs, num_machines, max_makespan, slack, resource_tightness):
    """
    Generate multiple scheduling instances with varying sigma distances.
    """
    dataset = []
    for _ in range(num_instances):
        duration_set_0 = np.random.randint(1, 10, num_jobs // num_machines)
        duration_set_1 = np.random.randint(1, 10, num_jobs // num_machines)

        # Generate jobs
        jobs, makespan = generate_jobs_from_duration_sets(
            num_jobs, num_machines, duration_set_0, duration_set_1, max_makespan, slack, resource_tightness
        )

        # Compute sigma distance
        sigma_distance = compute_sigma_distance(jobs, max_makespan)

        # Store the instance and its metadata
        dataset.append({"jobs": jobs, "makespan": makespan, "sigma_distance": sigma_distance})

    return dataset

# Example usage
num_instances = 10
num_jobs = 20
num_machines = 2
max_makespan = 50
slack = 0.2
resource_tightness = 5

dataset = generate_dataset(num_instances, num_jobs, num_machines, max_makespan, slack, resource_tightness)
for instance in dataset:
    print(f"Instance Sigma Distance: {instance['sigma_distance']}")

Instance Sigma Distance: -1.7528189375158045
Instance Sigma Distance: -1.7415292412933931
Instance Sigma Distance: -1.8015426229942746
Instance Sigma Distance: -1.80408518800749
Instance Sigma Distance: -1.7815461514919082
Instance Sigma Distance: -1.769011931284956
Instance Sigma Distance: -1.6241207031193157
Instance Sigma Distance: -2.002888561077362
Instance Sigma Distance: -1.893896380096151
Instance Sigma Distance: -1.8337504260933664


In [39]:
dataset

[{'jobs': {1: {'duration': 8,
    'release': 0,
    'deadline': 11,
    'machine': 0,
    'resources': 4},
   2: {'duration': 8,
    'release': 0,
    'deadline': 11,
    'machine': 0,
    'resources': 2},
   3: {'duration': 3,
    'release': 10,
    'deadline': 18,
    'machine': 0,
    'resources': 5},
   4: {'duration': 3,
    'release': 10,
    'deadline': 18,
    'machine': 0,
    'resources': 4},
   5: {'duration': 7,
    'release': 20,
    'deadline': 37,
    'machine': 0,
    'resources': 5},
   6: {'duration': 9,
    'release': 20,
    'deadline': 40,
    'machine': 0,
    'resources': 1},
   7: {'duration': 2,
    'release': 30,
    'deadline': 44,
    'machine': 0,
    'resources': 2},
   8: {'duration': 5,
    'release': 40,
    'deadline': 50,
    'machine': 0,
    'resources': 5},
   9: {'duration': 6,
    'release': 0,
    'deadline': 50,
    'machine': 0,
    'resources': 3},
   10: {'duration': 6,
    'release': 0,
    'deadline': 50,
    'machine': 0,
    'resources':